In [1]:
# Feature Engineering for Medicare Fraud Detection
# Create predictive features from claims and beneficiary data
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

print("="*70)
print("FEATURE ENGINEERING FOR FRAUD DETECTION")
print("="*70)

FEATURE ENGINEERING FOR FRAUD DETECTION


In [3]:
# =============================================================================
# PART 1: LOAD PROCESSED DATA
# =============================================================================

print("\n Loading Processed Data...")
PROJECT_ROOT = Path().resolve().parents[1]
data_path = PROJECT_ROOT / "data" / "processed" 
inpatient= pd.read_csv(data_path / 'Inpatient_clean.csv')
outpatient=pd.read_csv(data_path / 'Outpatient_clean.csv')
beneficiaries=pd.read_csv(data_path / 'Beneficiaries_combined.csv')

# Convert dates back to datetime
date_cols_ip=['CLM_FROM_DT', 'CLM_THRU_DT', 'CLM_ADMSN_DT', 'NCH_BENE_DSCHRG_DT']
date_cols_op = ['CLM_FROM_DT', 'CLM_THRU_DT']
date_cols_bene = ['BENE_BIRTH_DT', 'BENE_DEATH_DT']

for col in date_cols_ip:
    inpatient[col] = pd.to_datetime(inpatient[col], errors='coerce')

for col in date_cols_op:
    outpatient[col] = pd.to_datetime(outpatient[col], errors='coerce')

for col in date_cols_bene:
    beneficiaries[col] = pd.to_datetime(beneficiaries[col], errors='coerce')

print(f" Inpatient: {len(inpatient):,} claims")
print(f" Outpatient: {len(outpatient):,} claims")
print(f" Beneficiaries: {len(beneficiaries):,} records")


 Loading Processed Data...
 Inpatient: 66,773 claims
 Outpatient: 790,790 claims
 Beneficiaries: 343,644 records


In [4]:
# =============================================================================
# PART 2: PROVIDER-LEVEL FEATURES
# =============================================================================

print("\n" + "="*70)
print("PROVIDER-LEVEL FEATURES")
print("="*70)


# Combine inpatient and outpatient for provider analysis
inpatient['CLAIM_TYPE'] = 'IP'
outpatient['CLAIM_TYPE'] = 'OP'

all_claims = pd.concat([
    inpatient[['DESYNPUF_ID', 'CLM_ID', 'PRVDR_NUM', 'CLM_PMT_AMT', 'CLM_FROM_DT', 'CLAIM_TYPE']],
    outpatient[['DESYNPUF_ID', 'CLM_ID', 'PRVDR_NUM', 'CLM_PMT_AMT', 'CLM_FROM_DT', 'CLAIM_TYPE']]
], ignore_index=True)

print(f"\n Combined claims dataset: {len(all_claims):,} claims")

# Provider-level aggregations
provider_features = all_claims.groupby('PRVDR_NUM').agg({
    'CLM_ID': 'count',
    'CLM_PMT_AMT': ['mean', 'median', 'std', 'sum', 'min', 'max'],
    'DESYNPUF_ID': 'nunique'
}).reset_index()

provider_features.columns = [
    'PRVDR_NUM',
    'provider_claim_count',
    'provider_avg_payment',
    'provider_median_payment',
    'provider_std_payment',
    'provider_total_payment',
    'provider_min_payment',
    'provider_max_payment',
    'provider_unique_patients'
]

# Calculate coefficient of variation (payment variability indicator)
provider_features['provider_payment_cv'] = (
    provider_features['provider_std_payment'] / provider_features['provider_avg_payment']
)

# Claims per patient ratio (high = possible fraud)
provider_features['provider_claims_per_patient'] = (
    provider_features['provider_claim_count'] / provider_features['provider_unique_patients']
)

# Statistical risk scores
provider_features['provider_volume_percentile'] = (
    provider_features['provider_claim_count'].rank(pct=True)
)

provider_features['provider_avg_payment_percentile'] = (
    provider_features['provider_avg_payment'].rank(pct=True)
)

print("\n Provider features created:")
print(f"  - Claim volume statistics")
print(f"  - Payment statistics (mean, median, std, cv)")
print(f"  - Patient diversity metrics")
print(f"  - Risk percentiles")


PROVIDER-LEVEL FEATURES

 Combined claims dataset: 857,563 claims

 Provider features created:
  - Claim volume statistics
  - Payment statistics (mean, median, std, cv)
  - Patient diversity metrics
  - Risk percentiles


In [5]:
# =============================================================================
# PART 3: BENEFICIARY-LEVEL FEATURES
# =============================================================================

print("\n" + "="*70)
print("BENEFICIARY-LEVEL FEATURES")
print("="*70)

# Beneficiary claim aggregations
bene_features = all_claims.groupby('DESYNPUF_ID').agg({
    'CLM_ID': 'count',
    'CLM_PMT_AMT': ['mean', 'sum', 'std'],
    'PRVDR_NUM': 'nunique',
    'CLM_FROM_DT': ['min', 'max']
}).reset_index()

bene_features.columns = [
    'DESYNPUF_ID',
    'bene_claim_count',
    'bene_avg_payment',
    'bene_total_payment',
    'bene_std_payment',
    'bene_unique_providers',
    'bene_first_claim_date',
    'bene_last_claim_date'
]

# Days between first and last claim
bene_features['bene_claim_span_days'] = (
    (bene_features['bene_last_claim_date'] - bene_features['bene_first_claim_date']).dt.days
)

# Claims per day (utilization intensity)
bene_features['bene_claims_per_day'] = (
    bene_features['bene_claim_count'] / (bene_features['bene_claim_span_days'] + 1)
)

# Provider shopping behavior (many providers = potential fraud)
bene_features['bene_providers_per_claim'] = (
    bene_features['bene_unique_providers'] / bene_features['bene_claim_count']
)

# Separate IP and OP counts
ip_bene_counts = inpatient.groupby('DESYNPUF_ID').size().rename('bene_ip_claim_count')
op_bene_counts = outpatient.groupby('DESYNPUF_ID').size().rename('bene_op_claim_count')

bene_features = bene_features.merge(ip_bene_counts, on='DESYNPUF_ID', how='left')
bene_features = bene_features.merge(op_bene_counts, on='DESYNPUF_ID', how='left')

bene_features['bene_ip_claim_count'] = bene_features['bene_ip_claim_count'].fillna(0)
bene_features['bene_op_claim_count'] = bene_features['bene_op_claim_count'].fillna(0)

# IP to OP ratio
bene_features['bene_ip_to_op_ratio'] = (
    bene_features['bene_ip_claim_count'] / (bene_features['bene_op_claim_count'] + 1)
)

print("\n Beneficiary features created:")
print(f"  - Utilization statistics (claim counts, payment totals)")
print(f"  - Provider diversity (unique providers)")
print(f"  - Temporal patterns (claim span, intensity)")
print(f"  - IP/OP ratio")


BENEFICIARY-LEVEL FEATURES

 Beneficiary features created:
  - Utilization statistics (claim counts, payment totals)
  - Provider diversity (unique providers)
  - Temporal patterns (claim span, intensity)
  - IP/OP ratio


In [6]:

# =============================================================================
# PART 4: CHRONIC CONDITIONS & DEMOGRAPHICS
# =============================================================================

print("\n" + "="*70)
print("CHRONIC CONDITION & DEMOGRAPHIC FEATURES")
print("="*70)

# Use most recent year (2010) for each beneficiary
bene_2010 = beneficiaries[beneficiaries['YEAR'] == 2010].copy()

# Calculate age (as of 2010)
bene_2010['age'] = (
    (pd.to_datetime('2010-12-31') - bene_2010['BENE_BIRTH_DT']).dt.days / 365.25
).astype(int)

# Count chronic conditions
condition_cols = [
    'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD',
    'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS',
    'SP_RA_OA', 'SP_STRKETIA'
]

bene_2010['chronic_condition_count'] = (bene_2010[condition_cols] == 1).sum(axis=1)

# Create comorbidity flag
bene_2010['has_multiple_conditions'] = (bene_2010['chronic_condition_count'] >= 3).astype(int)

# High-cost conditions
bene_2010['has_high_cost_condition'] = (
    (bene_2010['SP_ALZHDMTA'] == 1) |
    (bene_2010['SP_CHF'] == 1) |
    (bene_2010['SP_CHRNKIDN'] == 1) |
    (bene_2010['SP_CNCR'] == 1)
).astype(int)

# Annual costs from beneficiary file
bene_2010['total_annual_cost'] = (
    bene_2010['MEDREIMB_IP'].fillna(0) + 
    bene_2010['MEDREIMB_OP'].fillna(0) + 
    bene_2010['MEDREIMB_CAR'].fillna(0)
)

# Select demographic features
demo_features = bene_2010[[
    'DESYNPUF_ID', 'age', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD',
    'chronic_condition_count', 'has_multiple_conditions', 'has_high_cost_condition',
    'total_annual_cost'
]].copy()

demo_features.columns = [
    'DESYNPUF_ID', 'age', 'gender', 'race',
    'chronic_condition_count', 'has_multiple_conditions', 'has_high_cost_condition',
    'bene_annual_cost_from_summary'
]

print("\n Demographic & clinical features created:")
print(f"  - Age")
print(f"  - Chronic condition count")
print(f"  - High-cost condition flags")
print(f"  - Annual cost from summary file")



CHRONIC CONDITION & DEMOGRAPHIC FEATURES

 Demographic & clinical features created:
  - Age
  - Chronic condition count
  - High-cost condition flags
  - Annual cost from summary file


In [7]:

# =============================================================================
# PART 5: CLAIM-LEVEL FEATURES
# =============================================================================

print("\n" + "="*70)
print("CLAIM-LEVEL FEATURES")
print("="*70)

def create_claim_features(df, claim_type='IP'):
    """Create features for individual claims"""
    df = df.copy()
    
    # Length of stay (inpatient only)
    if claim_type == 'IP':
        df['length_of_stay'] = (df['CLM_THRU_DT'] - df['CLM_FROM_DT']).dt.days + 1
        df['length_of_stay'] = df['length_of_stay'].fillna(0)
    
    # Days between admission and discharge (inpatient)
    if claim_type == 'IP':
        df['admission_to_discharge_days'] = (
            df['NCH_BENE_DSCHRG_DT'] - df['CLM_ADMSN_DT']
        ).dt.days
    
    # Payment per day
    if claim_type == 'IP':
        df['payment_per_day'] = df['CLM_PMT_AMT'] / (df['length_of_stay'] + 1)
    
    # Diagnosis code count (non-null diagnosis codes)
    dx_cols = [c for c in df.columns if 'ICD9_DGNS_CD' in c]
    df['diagnosis_code_count'] = df[dx_cols].notna().sum(axis=1)
    
    # Procedure code count
    proc_cols = [c for c in df.columns if 'ICD9_PRCDR_CD' in c]
    df['procedure_code_count'] = df[proc_cols].notna().sum(axis=1)
    
    # HCPCS code count
    hcpcs_cols = [c for c in df.columns if 'HCPCS_CD' in c]
    df['hcpcs_code_count'] = df[hcpcs_cols].notna().sum(axis=1)
    
    # Day of week (potential fraud pattern)
    df['claim_day_of_week'] = df['CLM_FROM_DT'].dt.dayofweek
    df['claim_is_weekend'] = (df['claim_day_of_week'] >= 5).astype(int)
    
    # Month (seasonality)
    df['claim_month'] = df['CLM_FROM_DT'].dt.month
    
    # Quarter
    df['claim_quarter'] = df['CLM_FROM_DT'].dt.quarter
    
    return df

# Apply to inpatient
print("\n Creating inpatient claim features...")
inpatient_featured = create_claim_features(inpatient, 'IP')

# Apply to outpatient
print(" Creating outpatient claim features...")
outpatient_featured = create_claim_features(outpatient, 'OP')

print("\n Claim-level features created:")
print(f"  - Length of stay (IP)")
print(f"  - Payment per day")
print(f"  - Diagnosis/procedure/HCPCS code counts")
print(f"  - Temporal features (day, month, quarter)")


CLAIM-LEVEL FEATURES

 Creating inpatient claim features...
 Creating outpatient claim features...

 Claim-level features created:
  - Length of stay (IP)
  - Payment per day
  - Diagnosis/procedure/HCPCS code counts
  - Temporal features (day, month, quarter)


In [8]:

# =============================================================================
# PART 6: MERGE ALL FEATURES
# =============================================================================

print("\n" + "="*70)
print("MERGING FEATURES")
print("="*70)

# Merge provider features to claims
print("\n Merging provider features...")
inpatient_featured = inpatient_featured.merge(
    provider_features, on='PRVDR_NUM', how='left'
)
outpatient_featured = outpatient_featured.merge(
    provider_features, on='PRVDR_NUM', how='left'
)

# Merge beneficiary features
print(" Merging beneficiary features...")
inpatient_featured = inpatient_featured.merge(
    bene_features.drop(['bene_first_claim_date', 'bene_last_claim_date'], axis=1),
    on='DESYNPUF_ID', how='left'
)
outpatient_featured = outpatient_featured.merge(
    bene_features.drop(['bene_first_claim_date', 'bene_last_claim_date'], axis=1),
    on='DESYNPUF_ID', how='left'
)

# Merge demographic features
print(" Merging demographic features...")
inpatient_featured = inpatient_featured.merge(
    demo_features, on='DESYNPUF_ID', how='left'
)
outpatient_featured = outpatient_featured.merge(
    demo_features, on='DESYNPUF_ID', how='left'
)

print(f"\n Final feature sets:")
print(f"  Inpatient: {inpatient_featured.shape[0]:,} claims × {inpatient_featured.shape[1]} features")
print(f"  Outpatient: {outpatient_featured.shape[0]:,} claims × {outpatient_featured.shape[1]} features")



MERGING FEATURES

 Merging provider features...
 Merging beneficiary features...
 Merging demographic features...

 Final feature sets:
  Inpatient: 66,773 claims × 124 features
  Outpatient: 790,790 claims × 115 features


In [9]:

# =============================================================================
# PART 7: STATISTICAL DERIVED FEATURES
# =============================================================================

print("\n" + "="*70)
print("STATISTICAL DERIVED FEATURES")
print("="*70)

def add_statistical_features(df):
    """Add z-scores and percentiles for fraud detection"""
    
    # Z-scores for payment amount
    df['payment_zscore'] = (
        (df['CLM_PMT_AMT'] - df['CLM_PMT_AMT'].mean()) / df['CLM_PMT_AMT'].std()
    )
    
    # Payment percentile
    df['payment_percentile'] = df['CLM_PMT_AMT'].rank(pct=True)
    
    # Deviation from provider average
    df['payment_deviation_from_provider_avg'] = (
        df['CLM_PMT_AMT'] - df['provider_avg_payment']
    )
    
    # Deviation from beneficiary average
    df['payment_deviation_from_bene_avg'] = (
        df['CLM_PMT_AMT'] - df['bene_avg_payment']
    )
    
    # Cost-to-condition ratio (is cost justified by conditions?)
    df['cost_per_condition'] = (
        df['CLM_PMT_AMT'] / (df['chronic_condition_count'] + 1)
    )
    
    return df

print("\n Calculating statistical features...")
inpatient_featured = add_statistical_features(inpatient_featured)
outpatient_featured = add_statistical_features(outpatient_featured)

print(" Statistical features added:")
print("  - Z-scores for outlier detection")
print("  - Percentile rankings")
print("  - Deviations from averages")
print("  - Cost-to-condition ratios")


STATISTICAL DERIVED FEATURES

 Calculating statistical features...
 Statistical features added:
  - Z-scores for outlier detection
  - Percentile rankings
  - Deviations from averages
  - Cost-to-condition ratios


In [10]:

# =============================================================================
# PART 8: SAVE FEATURED DATASETS
# =============================================================================

print("\n" + "="*70)
print("SAVING FEATURED DATASETS")
print("="*70)

# Save full featured datasets
inpatient_featured.to_csv( PROJECT_ROOT / "data" / "processed" /'inpatient_featured.csv', index=False)
outpatient_featured.to_csv( PROJECT_ROOT / "data" / "processed" /'outpatient_featured.csv', index=False)

print(f"\n Saved featured datasets:")
print(f"  - inpatient_featured.csv ({len(inpatient_featured):,} rows)")
print(f"  - outpatient_featured.csv ({len(outpatient_featured):,} rows)")



SAVING FEATURED DATASETS

 Saved featured datasets:
  - inpatient_featured.csv (66,773 rows)
  - outpatient_featured.csv (790,790 rows)


In [11]:

# =============================================================================
# PART 9: FEATURE SUMMARY
# =============================================================================

print("\n" + "="*70)
print("FEATURE ENGINEERING SUMMARY")
print("="*70)

feature_categories = {
    'Provider Features': [
        'provider_claim_count', 'provider_avg_payment', 'provider_payment_cv',
        'provider_claims_per_patient', 'provider_volume_percentile'
    ],
    'Beneficiary Features': [
        'bene_claim_count', 'bene_total_payment', 'bene_unique_providers',
        'bene_claims_per_day', 'bene_ip_to_op_ratio'
    ],
    'Clinical Features': [
        'chronic_condition_count', 'has_multiple_conditions',
        'has_high_cost_condition', 'diagnosis_code_count', 'procedure_code_count'
    ],
    'Claim Features': [
        'CLM_PMT_AMT', 'length_of_stay', 'payment_per_day',
        'claim_day_of_week', 'claim_month'
    ],
    'Statistical Features': [
        'payment_zscore', 'payment_percentile',
        'payment_deviation_from_provider_avg', 'cost_per_condition'
    ]
}

print("\n FEATURE CATEGORIES:")
for category, features in feature_categories.items():
    print(f"\n{category}:")
    for feat in features:
        if feat in inpatient_featured.columns:
            print(f"   {feat}")

print(f"\n Total features created: {inpatient_featured.shape[1]}")
print(f"   Ready for fraud label creation and modeling!")

print("\n" + "="*70)
print(" FEATURE ENGINEERING COMPLETE")
print("="*70)


FEATURE ENGINEERING SUMMARY

 FEATURE CATEGORIES:

Provider Features:
   provider_claim_count
   provider_avg_payment
   provider_payment_cv
   provider_claims_per_patient
   provider_volume_percentile

Beneficiary Features:
   bene_claim_count
   bene_total_payment
   bene_unique_providers
   bene_claims_per_day
   bene_ip_to_op_ratio

Clinical Features:
   chronic_condition_count
   has_multiple_conditions
   has_high_cost_condition
   diagnosis_code_count
   procedure_code_count

Claim Features:
   CLM_PMT_AMT
   length_of_stay
   payment_per_day
   claim_day_of_week
   claim_month

Statistical Features:
   payment_zscore
   payment_percentile
   payment_deviation_from_provider_avg
   cost_per_condition

 Total features created: 129
   Ready for fraud label creation and modeling!

 FEATURE ENGINEERING COMPLETE
